# Extra · Visão computacional: da convolução à detecção
**Aprendizagem Computacional · 4º TIA · Apostila do Aluno · notebook extra · versão INICIAL**

Complemento da seção *Redes neurais para imagens* do **cap. 15**. Pré-requisito: o notebook extra de redes neurais.

| Seção | O que você vai ver | Apostila |
|---|---|---|
| 1 | Imagem é uma matriz de números | cap. 2, 15 |
| 2 | Convolução feita à mão: filtros que detectam bordas | cap. 15 |
| 3 | O problema do MLP: deslocar a imagem 2 pixels derruba o acerto | cap. 15 |
| 4 | Uma rede convolucional (CNN) pequena em PyTorch, contra o MLP | cap. 9, 15 |
| 5 | Métricas de detecção: IoU, precisão, recall e AP, calculados à mão | cap. 12, 15 |
| 6 | (Opcional) YOLO pré-treinado: o baseline de um projeto de detecção | cap. 15 |
| 7 | Placas de veículo em 882 fotos reais: regras escritas à mão × detector treinado | cap. 15 e 24 |

As seções 1 a 5 rodam sem internet, em CPU, em poucos minutos. A seção 6 precisa de internet e fica desligada por padrão.

**Como usar a versão INICIAL:** há 4 lacunas `# TODO`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEMENTE = 42
np.random.seed(SEMENTE)

## 1 · Imagem é uma matriz
Uma foto em tons de cinza é uma tabela: cada célula é um pixel, de 0 (preto) a 255 (branco). Uma foto colorida são 3 tabelas empilhadas (vermelho, verde, azul). Para o computador, \"ver\" é fazer contas com essas tabelas.

In [ ]:
from skimage import data
foto = data.camera()            # imagem de exemplo que vem com o scikit-image
print('Formato:', foto.shape, '| tipo:', foto.dtype, '| mínimo e máximo:', foto.min(), foto.max())
fig, eixos = plt.subplots(1, 2, figsize=(10, 4))
eixos[0].imshow(foto, cmap='gray'); eixos[0].set_title('A imagem')
recorte = foto[200:210, 200:210]
eixos[1].imshow(recorte, cmap='gray')
for (i, j), v in np.ndenumerate(recorte):
    eixos[1].text(j, i, v, ha='center', va='center', fontsize=7, color='red')
eixos[1].set_title('Um recorte 10 × 10: são só números')
for ax in eixos: ax.axis('off')
plt.tight_layout(); plt.show()

## 2 · Convolução à mão
Um **filtro** (ou *kernel*) é uma pequena matriz, aqui 3 × 3. A convolução desliza o filtro pela imagem e, em cada posição, multiplica e soma. O filtro de Sobel abaixo responde forte onde a imagem muda de claro para escuro na horizontal: ele é um **detector de bordas verticais**.

Numa CNN, ninguém escolhe os filtros: a rede **aprende** os valores, do mesmo jeito que a regressão aprende os coeficientes.

In [ ]:
def convolucao(imagem, filtro):
    h, w = filtro.shape
    saida = np.zeros((imagem.shape[0] - h + 1, imagem.shape[1] - w + 1))
    for i in range(saida.shape[0]):
        for j in range(saida.shape[1]):
            # TODO: multiplique o pedaço imagem[i:i+h, j:j+w] pelo filtro, elemento a elemento, e some tudo
            saida[i, j] = ...
    return saida

sobel_vertical = np.array([[-1, 0, 1],
                           [-2, 0, 2],
                           [-1, 0, 1]])
sobel_horizontal = sobel_vertical.T

pequena = foto[::4, ::4].astype(float)        # reduz para rodar rápido em Python puro
bordas_v = convolucao(pequena, sobel_vertical)
bordas_h = convolucao(pequena, sobel_horizontal)
fig, eixos = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, tit in zip(eixos, [pequena, np.abs(bordas_v), np.abs(bordas_h)],
                        ['original', 'filtro de bordas verticais', 'filtro de bordas horizontais']):
    ax.imshow(img, cmap='gray'); ax.set_title(tit); ax.axis('off')
plt.tight_layout(); plt.show()

**Pergunta para discussão:** que filtro você usaria para achar os contornos retangulares de uma placa de veículo?

**Resposta esperada:** uma combinação de bordas verticais e horizontais. É exatamente o que as primeiras camadas de uma CNN aprendem sozinhas. As camadas seguintes combinam bordas em cantos, cantos em retângulos e caracteres, e assim por diante.

## 3 · O problema do MLP: deslocar 2 pixels
Treinamos um MLP nos dígitos 8 × 8 (como no notebook de redes neurais) e testamos em duas versões do teste: a original e a mesma imagem **deslocada 2 pixels** para a direita. Para uma pessoa, é o mesmo dígito.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

dig = load_digits()
imagens, y = dig.images / 16.0, dig.target
img_tr, img_te, y_tr, y_te = train_test_split(imagens, y, test_size=0.25, stratify=y, random_state=SEMENTE)

def deslocar(lote, px=2):
    saida = np.zeros_like(lote)
    saida[:, :, px:] = lote[:, :, :-px]
    return saida

img_te_desl = deslocar(img_te)
mlp = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-3, max_iter=1000, random_state=SEMENTE))
mlp.fit(img_tr.reshape(len(img_tr), -1), y_tr)
acc_mlp = mlp.score(img_te.reshape(len(img_te), -1), y_te)
acc_mlp_desl = mlp.score(img_te_desl.reshape(len(img_te), -1), y_te)
print(f'MLP · teste original: {acc_mlp:.3f} | teste deslocado 2 px: {acc_mlp_desl:.3f}')

fig, eixos = plt.subplots(1, 2, figsize=(4, 2))
eixos[0].imshow(img_te[0], cmap='gray_r'); eixos[0].set_title('original')
eixos[1].imshow(img_te_desl[0], cmap='gray_r'); eixos[1].set_title('deslocada')
for ax in eixos: ax.axis('off')
plt.show()

**Leitura:** o acerto despenca. Para o MLP, cada pixel é uma coluna fixa: se o traço muda de coluna, é outro dado. Nas câmeras, o veículo nunca está no mesmo lugar; precisamos de um modelo que reconheça o padrão **onde quer que ele esteja**.

## 4 · Uma CNN pequena em PyTorch
Arquitetura: convolução 3 × 3 (16 filtros) → ReLU → *pooling* (resume cada região 2 × 2 pelo maior valor) → convolução (32 filtros) → ReLU → camada final com 10 saídas. Mesmo split, mesmo teste original e deslocado. Como o modelo é pequeno, roda em CPU em segundos.

Para ficar justo com o MLP, treinamos a CNN **sem** imagens deslocadas: ela também nunca viu um dígito fora do lugar.

In [ ]:
import torch
from torch import nn
torch.manual_seed(SEMENTE)

def tensor(lote):
    return torch.tensor(lote, dtype=torch.float32).unsqueeze(1)   # (n, 1 canal, 8, 8)

Xtr, ytr = tensor(img_tr), torch.tensor(y_tr)
cnn = nn.Sequential(
    # TODO: primeira convolução: 1 canal de entrada, 16 filtros, kernel_size=3, padding=1
    ...,
    nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.AdaptiveMaxPool2d(1),
    nn.Flatten(), nn.Linear(32, 10))
otimizador = torch.optim.Adam(cnn.parameters(), lr=0.01)
perda_fn = nn.CrossEntropyLoss()

historico = []
for epoca in range(60):
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 64):
        lote = perm[i:i + 64]
        otimizador.zero_grad()
        perda = perda_fn(cnn(Xtr[lote]), ytr[lote])
        perda.backward(); otimizador.step()
    historico.append(perda.item())

def acuracia(modelo, lote, alvo):
    with torch.no_grad():
        return (modelo(tensor(lote)).argmax(1).numpy() == alvo).mean()

acc_cnn, acc_cnn_desl = acuracia(cnn, img_te, y_te), acuracia(cnn, img_te_desl, y_te)
print('Parâmetros: CNN =', sum(p.numel() for p in cnn.parameters()),
      '| MLP =', sum(w.size for w in mlp[-1].coefs_) + sum(b.size for b in mlp[-1].intercepts_))
pd.DataFrame({'teste original': [acc_mlp, acc_cnn], 'teste deslocado 2 px': [acc_mlp_desl, acc_cnn_desl]},
             index=['MLP', 'CNN']).round(3)

In [ ]:
filtros = cnn[0].weight.detach().numpy()[:, 0]
fig, eixos = plt.subplots(2, 8, figsize=(10, 2.8))
for ax, f in zip(eixos.ravel(), filtros):
    ax.imshow(f, cmap='RdBu'); ax.axis('off')
plt.suptitle('Os 16 filtros 3 × 3 que a CNN aprendeu sozinha (compare com o Sobel da seção 2)'); plt.show()

**Leitura:** com um número parecido de parâmetros, a CNN acerta quase tanto quanto o MLP nas imagens originais (cerca de 97% contra 98%) e resiste muito melhor ao deslocamento (cerca de 74% contra 15%). Duas ideias explicam: o **mesmo filtro** percorre a imagem inteira (o padrão é reconhecido em qualquer posição) e o *pooling* resume regiões (pequenos deslocamentos mudam pouco o resumo).

Nas câmeras reais, as imagens têm centenas de pixels e milhões de exemplos de treino seriam necessários para treinar do zero. Por isso usamos redes **pré-treinadas** e fazemos ajuste fino (seção 6 e cap. 15).

## 5 · Métricas de detecção à mão
Um detector devolve **caixas** (x1, y1, x2, y2) com uma confiança. Para saber se uma caixa acertou, comparamos com a caixa verdadeira usando o **IoU**: área da interseção ÷ área da união.

In [ ]:
def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    intersecao = max(0, x2 - x1) * max(0, y2 - y1)
    area = lambda c: (c[2] - c[0]) * (c[3] - c[1])
    uniao = ...   # TODO: área da união = área(a) + área(b) − interseção
    return intersecao / uniao

verdade = (100, 100, 200, 160)
for nome, caixa in [('quase perfeita', (105, 102, 203, 162)), ('deslocada', (140, 110, 240, 170)), ('errada', (300, 300, 350, 330))]:
    print(f'{nome:15s} IoU = {iou(verdade, caixa):.2f}')

Agora um cenário de 3 imagens com placas verdadeiras e as caixas que um detector previu. Uma previsão é **verdadeiro positivo** se o IoU com uma placa ainda não encontrada for ≥ 0,5. Ordenando as previsões pela confiança, calculamos precisão e recall acumulados: a área sob essa curva é o **AP** (*Average Precision*). A média do AP entre classes é o **mAP**.

In [ ]:
placas = {1: [(10, 10, 60, 30), (100, 50, 150, 70)], 2: [(20, 40, 70, 60)], 3: [(30, 30, 80, 50), (120, 10, 170, 30)]}
previsoes = [  # (imagem, caixa, confiança)
    (1, (12, 11, 61, 31), 0.95), (2, (22, 41, 71, 62), 0.90), (3, (31, 29, 79, 51), 0.85),
    (1, (200, 200, 240, 220), 0.70), (3, (125, 12, 172, 33), 0.60), (2, (60, 40, 110, 60), 0.40),
]
encontradas, linhas = set(), []
total_placas = sum(len(v) for v in placas.values())
vp = fp = 0
for img, caixa, conf in sorted(previsoes, key=lambda p: -p[2]):
    melhor = max(range(len(placas[img])), key=lambda k: iou(caixa, placas[img][k]))
    acerto = iou(caixa, placas[img][melhor]) >= 0.5 and (img, melhor) not in encontradas
    if acerto:
        encontradas.add((img, melhor)); vp += 1
    else:
        fp += 1
    linhas.append({'confiança': conf, 'acerto': acerto, 'precisão': vp / (vp + fp), 'recall': vp / total_placas})
tabela = pd.DataFrame(linhas)
ap = sum((tabela['recall'].diff().fillna(tabela['recall'].iloc[0])) * tabela['precisão'])
print(f'Placas verdadeiras: {total_placas} | AP@0,5 ≈ {ap:.2f}')
tabela.round(2)

**Leitura:** a placa (100, 50, 150, 70) da imagem 1 nunca foi encontrada, então o recall máximo é 4/5 = 0,8. A previsão de confiança 0,70 é um alarme falso. Esse é exatamente o cálculo que o YOLO faz por você no `model.val()`, em milhares de caixas.

**Pergunta para discussão:** num sistema de fiscalização, você prefere subir a confiança mínima (menos alarmes falsos) ou baixar (menos placas perdidas)?

**Resposta esperada:** depende do custo de cada erro, como no limiar do cap. 12. Se cada alarme gera uma multa, alarmes falsos são graves e a confiança deve subir. Se o objetivo é só contar o fluxo, perder algumas placas importa menos que contar errado.

## 6 · (Opcional) YOLO pré-treinado
Num projeto real, você não treina do zero: parte de um detector pré-treinado (o **baseline**), mede no **seu** teste e depois faz ajuste fino. Para rodar no Colab, mude `RODAR_YOLO` para `True`. Precisa de internet; com GPU (*Ambiente de execução → Alterar tipo → T4*) fica mais rápido.

O protocolo do cap. 15 continua valendo: split por câmera ou por dia, teste anotado pelo grupo e usado uma vez, análise de erros por condição e cuidado com a LGPD.

In [ ]:
RODAR_YOLO = False

if RODAR_YOLO:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLO
    modelo = YOLO('yolov8n.pt')                                   # pré-treinado em 80 classes (COCO)
    resultado = modelo('https://ultralytics.com/images/bus.jpg')  # imagem de exemplo da biblioteca
    resultado[0].show()
    for caixa in resultado[0].boxes:
        print(modelo.names[int(caixa.cls)], round(float(caixa.conf), 2), [round(v) for v in caixa.xyxy[0].tolist()])
    # Projeto: modelo.val(data='placas.yaml') -> baseline; modelo.train(data='placas.yaml', epochs=50, seed=42) -> ajuste fino
else:
    print('Seção opcional desligada. Mude RODAR_YOLO para True no Colab (precisa de internet).')

## 7 · Exemplo com placa de veículo, em fotos reais
Leitura de placa é um **pipeline** de etapas, cada uma com seu próprio erro (cap. 15):

**localizar a placa → recortar → ler (OCR)**

Esta seção trata da etapa mais difícil e que decide todo o resto: **localizar**. Vamos usar uma base pública com fotos reais e caixas anotadas à mão:

- **Vehicle Registration Plates** (Roboflow / Augmented Startups, publicada no Hugging Face como `keremberke/license-plate-object-detection`), licença **CC BY 4.0**;
- usaremos a parte de **teste**: 882 fotos de carros e motos, com 902 placas anotadas no formato COCO;
- as placas são de vários países. Bases brasileiras (RodoSol-ALPR, UFPR-ALPR, da UFPR) são liberadas para pesquisa mediante pedido com e-mail institucional.

**LGPD:** esta base é pública e licenciada para uso. Imagens de câmeras da SINFRA **não** são: elas só podem ser usadas com autorização e não podem ser republicadas (cap. 15).

A pergunta da seção: **um método com regras escritas à mão resolve o problema, ou precisamos de um detector treinado?**

In [ ]:
import os, json, zipfile, urllib.request, cv2

URL_PLACAS = 'https://huggingface.co/datasets/keremberke/license-plate-object-detection/resolve/main/data/test.zip'
# Tudo que esta seção baixa e gera fica FORA da pasta do projeto (/content no Colab, pasta do usuário no laboratório)
from pathlib import Path
TRABALHO = (Path('/content') if Path('/content').exists() else Path.home() / 'ac_saida') / 'visao_placas'
TRABALHO.mkdir(parents=True, exist_ok=True)
os.chdir(TRABALHO)
print('Arquivos baixados e gerados vão para:', TRABALHO)
PASTA_PLACAS = os.environ.get('PASTA_PLACAS', 'placas_teste')

if not os.path.exists(os.path.join(PASTA_PLACAS, '_annotations.coco.json')):
    print('Baixando a base (22 MB)...')
    urllib.request.urlretrieve(URL_PLACAS, 'placas_teste.zip')
    zipfile.ZipFile('placas_teste.zip').extractall(PASTA_PLACAS)

coco = json.load(open(os.path.join(PASTA_PLACAS, '_annotations.coco.json')))
caixas_reais = {}                                  # id da imagem -> lista de caixas (x1, y1, x2, y2)
for a in coco['annotations']:
    x, y, w, h = a['bbox']                         # COCO guarda (x, y, largura, altura)
    caixas_reais.setdefault(a['image_id'], []).append((x, y, x + w, y + h))
imagens = coco['images']
print(f'{len(imagens)} imagens | {len(coco["annotations"])} placas anotadas')

def ler(im):
    return cv2.cvtColor(cv2.imread(os.path.join(PASTA_PLACAS, im['file_name'])), cv2.COLOR_BGR2RGB)

def desenhar(img, caixas, cor):
    img = img.copy()
    for x1, y1, x2, y2 in caixas:
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), cor, 3)
    return img

amostra = np.random.default_rng(SEMENTE).choice(len(imagens), 8, replace=False)
fig, eixos = plt.subplots(2, 4, figsize=(14, 6))
for ax, i in zip(eixos.ravel(), amostra):
    ax.imshow(desenhar(ler(imagens[i]), caixas_reais[imagens[i]['id']], (0, 255, 0))); ax.axis('off')
plt.suptitle('Amostra da base, com a placa anotada em verde'); plt.tight_layout(); plt.show()

**Repare na variedade:** carros e motos, de frente e de lado, de dia e de noite, placas retangulares e quadradas, perto e longe. É essa variedade que um sistema na rua vai enfrentar. Boa parte das fotos vem de câmeras de entrada de estacionamento, um cenário parecido com o de uma câmera fixa de trânsito. Algumas fotos têm bordas "esticadas", um preenchimento feito na preparação da base: bases públicas também têm defeitos, e isso faz parte do trabalho.

### 7.1 · Método clássico: bordas + regras de formato
A ideia é a mesma da seção 2: achar bordas (Canny), juntar as bordas em contornos e ficar com o retângulo que **tem cara de placa**. "Cara de placa" é uma regra que nós escrevemos: a placa de carro Mercosul mede 40 × 13 cm, então a largura é cerca de 3 vezes a altura.

Para avaliar, usamos a métrica da seção 5: para cada foto, o método aponta **uma** caixa, e ela conta como acerto se tiver **IoU ≥ 0,5** com alguma placa anotada.

In [ ]:
def detectar_classico(img, razao_min=2.5, razao_max=4.5, area_min=0.002):
    cinza = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY); H, W = cinza.shape
    bordas = cv2.Canny(cv2.GaussianBlur(cinza, (5, 5), 0), 50, 150)
    contornos, _ = cv2.findContours(bordas, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    candidatas = []
    for x, y, w, h in map(cv2.boundingRect, contornos):
        # TODO: guarde só os retângulos com cara de placa:
        #       razão w/h entre razao_min e razao_max, e área w*h entre area_min*H*W e 0.5*H*W
        if ...:
            candidatas.append((w * h, (x, y, x + w, y + h)))
    return max(candidatas)[1] if candidatas else None       # fica com a maior candidata

def avaliar(detector):
    linhas = []
    for im in imagens:
        img = ler(im)
        prevista = detector(img)
        reais = caixas_reais[im['id']]
        melhor = max(iou(prevista, r) for r in reais) if prevista is not None else 0.0
        x1, y1, x2, y2 = reais[0]
        linhas.append({'arquivo': im['file_name'], 'prevista': prevista, 'IoU': melhor, 'acerto': melhor >= 0.5,
                       'razão real (w/h)': (x2 - x1) / max(1, y2 - y1),
                       'tamanho da placa (% da foto)': 100 * (x2 - x1) * (y2 - y1) / (im['width'] * im['height']),
                       'brilho médio': img.mean()})
    return pd.DataFrame(linhas)

res_classico = avaliar(detectar_classico)
print(f'Método clássico: acerto (IoU ≥ 0,5) em {res_classico.acerto.mean():.1%} das fotos | IoU médio {res_classico.IoU.mean():.2f}')

### 7.2 · Diagnóstico: por que errou tanto?
Antes de mexer no método, olhamos os erros (cap. 15: a análise de erros vem antes do ajuste). Dois gráficos e uma galeria:

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 3.8))
eixos[0].hist(res_classico['razão real (w/h)'].clip(0, 7), bins=35, color='#1F5F3A')
eixos[0].axvspan(2.5, 4.5, color='orange', alpha=0.25, label='faixa da nossa regra')
eixos[0].set_title('Formato real das placas (largura / altura)'); eixos[0].legend()
eixos[1].hist(res_classico['tamanho da placa (% da foto)'].clip(0, 15), bins=35, color='#1F5F3A')
eixos[1].set_title('Tamanho da placa (% da área da foto)')
plt.tight_layout(); plt.show()
dentro = res_classico['razão real (w/h)'].between(2.5, 4.5).mean()
print(f'Só {dentro:.0%} das placas reais têm o formato que a regra exige.')

erros = res_classico[~res_classico.acerto].sample(8, random_state=SEMENTE)
fig, eixos = plt.subplots(2, 4, figsize=(14, 6))
for ax, (_, linha) in zip(eixos.ravel(), erros.iterrows()):
    im = next(i for i in imagens if i['file_name'] == linha.arquivo)
    img = desenhar(ler(im), caixas_reais[im['id']], (0, 255, 0))
    if linha.prevista is not None:
        img = desenhar(img, [linha.prevista], (255, 200, 0))
    ax.imshow(img); ax.axis('off')
plt.suptitle('Erros do método clássico (verde: placa real | amarelo: o que o método escolheu)'); plt.tight_layout(); plt.show()

**Leitura:**
1. **Formato:** muitas placas não têm a proporção 3:1. Placas de moto e vários padrões estrangeiros são quase quadrados, e a foto em ângulo deforma a proporção. A regra descarta a placa antes de qualquer coisa.
2. **Escolha da candidata:** "fica com a maior" escolhe cancelas, faixas pintadas no chão e vidros do carro, que também são retângulos com bordas fortes.
3. **Tamanho:** a placa ocupa, na mediana, cerca de 2% da foto. Um retângulo qualquer do fundo facilmente é maior.

### 7.3 · Melhorando as regras (e vendo o limite delas)
Usamos o diagnóstico para escrever regras melhores: aceitar placas quadradas (razão de 1 a 6), limitar o tamanho máximo e, em vez de ficar com a maior candidata, ficar com a de **maior densidade de bordas** (os caracteres criam muitas bordas dentro da placa).

In [ ]:
def detectar_classico_v2(img):
    cinza = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY); H, W = cinza.shape
    bordas = cv2.Canny(cv2.GaussianBlur(cinza, (5, 5), 0), 50, 150)
    contornos, _ = cv2.findContours(bordas, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    melhor, melhor_densidade = None, 0
    for x, y, w, h in map(cv2.boundingRect, contornos):
        if h == 0 or not (1.0 <= w / h <= 6.0) or not (0.003 * H * W < w * h < 0.15 * H * W):
            continue
        densidade = bordas[y:y + h, x:x + w].mean() / 255      # fração de pixels de borda dentro da caixa
        if densidade > melhor_densidade:
            melhor, melhor_densidade = (x, y, x + w, y + h), densidade
    return melhor

res_v2 = avaliar(detectar_classico_v2)
print(f'Clássico v1: {res_classico.acerto.mean():.1%} | clássico v2: {res_v2.acerto.mean():.1%} de acerto')

# Desempenho POR CONDIÇÃO (é assim que deve aparecer no artigo, cap. 24)
tamanho = pd.qcut(res_classico['tamanho da placa (% da foto)'], 3, labels=['pequena', 'média', 'grande'])
luz = pd.qcut(res_classico['brilho médio'], 3, labels=['escura', 'média', 'clara'])
for nome, grupo in [('tamanho da placa', tamanho), ('luz da foto', luz)]:
    tabela = pd.DataFrame({'clássico v1': res_classico.groupby(grupo, observed=True).acerto.mean(),
                           'clássico v2': res_v2.groupby(grupo, observed=True).acerto.mean()})
    tabela.index.name = nome
    display(tabela.style.format('{:.1%}'))

**Leitura:** a v2 melhora a média, mas olhe a tabela por condição. A v1 vai mal em tudo (de 2% a 7%). A v2 dá um salto nas placas **pequenas** (32%), mas quase não acerta as **grandes** (1,4%, pior que a v1): regiões pequenas e cheias de detalhe vencem a placa grande no critério de densidade de bordas. Nas fotos escuras, a v2 também fica bem abaixo do que consegue com luz média ou clara. Cada regra nova conserta um tipo de erro e cria outro: escrever regras para todos os casos é inviável. E, se tivéssemos olhado só a média, não teríamos visto essa troca.

**Esta é a virada do capítulo 15:** em vez de escrever as regras, **aprendemos** os filtros a partir de exemplos anotados, como a CNN da seção 4. Um detector como o YOLO, ajustado (fine-tuning) com milhares de placas anotadas, aprende sozinho que placa pode ser quadrada, pequena, escura ou inclinada.

### 7.4 · (Opcional, Colab) Treinar um YOLO para placas e comparar
A célula abaixo ajusta o YOLOv8n pré-treinado com uma **amostra** da parte de treino da base e avalia nas **mesmas 882 fotos de teste**, com a **mesma função `avaliar`**. Assim a comparação com o método clássico é justa.

Treinar com as 6.176 fotos, 20 épocas, em 640 px leva cerca de **1h30 por época na CPU** do Colab. Por isso a célula escolhe sozinha uma configuração reduzida, conforme o ambiente:

| Ambiente | Fotos de treino | Épocas | Resolução | Tempo aproximado |
|---|---|---|---|---|
| CPU (Colab gratuito sem GPU) | amostra de 600 | 3 | 320 px | 5 a 8 min |
| GPU (Ambiente de execução → Alterar o tipo de ambiente) | todas (6.176) | 20 | 640 px | cerca de 10 min numa A100; numa T4, algumas vezes mais |

Isso funciona porque o modelo já vem **pré-treinado** (transfer learning, cap. 15): ele já sabe achar bordas, cantos e texturas, e só precisa aprender o que é uma placa. Na base completa, o mAP50 na validação já passa de 0,9 depois de uma única época. Para rodar, mude `RODAR_TREINO_PLACAS` para `True`.

In [ ]:
RODAR_TREINO_PLACAS = False

def coco_para_yolo(pasta_coco, pasta_saida, ids):
    """Converte as anotações COCO (x, y, w, h em pixels) das imagens em `ids` para o formato YOLO
    (classe, centro e tamanho de 0 a 1)."""
    import shutil
    d = json.load(open(os.path.join(pasta_coco, '_annotations.coco.json')))
    os.makedirs(f'{pasta_saida}/images', exist_ok=True); os.makedirs(f'{pasta_saida}/labels', exist_ok=True)
    por_imagem = {}
    for a in d['annotations']:
        por_imagem.setdefault(a['image_id'], []).append(a['bbox'])
    for im in d['images']:
        if im['id'] not in ids:
            continue
        W, H = im['width'], im['height']
        shutil.copy(os.path.join(pasta_coco, im['file_name']), f'{pasta_saida}/images/')
        with open(f"{pasta_saida}/labels/{os.path.splitext(im['file_name'])[0]}.txt", 'w') as f:
            for x, y, w, h in por_imagem.get(im['id'], []):
                f.write(f'0 {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}\n')

if RODAR_TREINO_PLACAS:
    import subprocess, sys, time, torch
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLO

    TEM_GPU = torch.cuda.is_available()

    def baixar(parte):
        if not os.path.exists(f'placas_{parte}/_annotations.coco.json'):
            urllib.request.urlretrieve(URL_PLACAS.replace('test.zip', f'{parte}.zip'), f'placas_{parte}.zip')
            zipfile.ZipFile(f'placas_{parte}.zip').extractall(f'placas_{parte}')
        return [im['id'] for im in json.load(open(f'placas_{parte}/_annotations.coco.json'))['images']]

    # As 882 fotos de teste continuam intocadas até a avaliação final (cap. 9).
    if TEM_GPU:      # base completa: partes de treino (6.176) e validação (1.765)
        EPOCAS, TAMANHO = 20, 640
        coco_para_yolo('placas_train', 'yolo_placas/train', set(baixar('train')))
        coco_para_yolo('placas_valid', 'yolo_placas/val', set(baixar('valid')))
    else:            # sem GPU: amostras SEM sobreposição da parte de treino
        EPOCAS, TAMANHO = 3, 320
        ids = np.random.default_rng(SEMENTE).permutation(baixar('train'))
        coco_para_yolo('placas_train', 'yolo_placas/train', set(ids[:600].tolist()))
        coco_para_yolo('placas_train', 'yolo_placas/val', set(ids[600:750].tolist()))
    N_TREINO = len(os.listdir('yolo_placas/train/images'))
    print(f'GPU: {TEM_GPU} | {N_TREINO} fotos de treino, {EPOCAS} épocas, {TAMANHO} px')
    with open('placas.yaml', 'w') as f:
        f.write(f"path: {os.path.abspath('yolo_placas')}\ntrain: train/images\nval: val/images\nnames:\n  0: placa\n")

    inicio = time.time()
    modelo = YOLO('yolov8n.pt')                       # pré-treinado no COCO: transfer learning (cap. 15)
    modelo.train(data='placas.yaml', epochs=EPOCAS, imgsz=TAMANHO, batch=16, seed=SEMENTE,
                 plots=False, verbose=False)
    print(f'Treino: {(time.time() - inicio) / 60:.1f} min')

    def detectar_yolo(img):
        caixas = modelo.predict(img, imgsz=TAMANHO, conf=0.1, verbose=False)[0].boxes
        if len(caixas) == 0:
            return None
        return tuple(caixas.xyxy[caixas.conf.argmax()].tolist())      # a caixa de maior confiança

    res_yolo = avaliar(detectar_yolo)
    display(pd.DataFrame({'método': ['clássico v1', 'clássico v2', f'YOLOv8n ajustado ({N_TREINO} fotos, {EPOCAS} épocas)'],
                          'acerto (IoU ≥ 0,5)': [res_classico.acerto.mean(), res_v2.acerto.mean(), res_yolo.acerto.mean()]}))
else:
    print('Treino opcional desligado. No Colab, mude RODAR_TREINO_PLACAS para True (5 a 8 min na CPU).')

**Resultado de referência** (rodado pelo professor numa GPU A100, com a base completa: 20 épocas, cerca de 10 minutos; mAP50 de 0,98 na validação):

| Método | Acerto nas 882 fotos de teste (IoU ≥ 0,5) |
|---|---|
| Clássico v1 (regras de formato) | 3,7% |
| Clássico v2 (regras ajustadas pelo diagnóstico) | 13,6% |
| YOLOv8n ajustado | **98,9%** |

**Leitura:** as mesmas fotos e a mesma métrica, com resultados muito diferentes. Horas ajustando regras levaram de 3,7% a 13,6%; dez minutos de treino de um modelo pré-treinado chegaram a quase 99%. Esse é o argumento central do cap. 15. Mas atenção: 98,9% é o acerto nesta base, com fotos parecidas entre treino e teste. Numa câmera nova, de outro lugar e com outra luz, o número deve cair, e por isso a validação de um projeto real separa as imagens por câmera ou por dia.

O projeto real fica assim:

| Etapa | Método clássico (7.1 a 7.3) | Método aprendido (projeto) | Métrica |
|---|---|---|---|
| Localizar a placa | Bordas + regras de formato | YOLO ajustado com placas anotadas | IoU, precisão, recall, mAP |
| Ler | (não feito aqui) | OCR (EasyOCR, PaddleOCR) ou um segundo YOLO só para caracteres | Acerto da placa inteira e por caractere |

**Sobre o OCR (célula opcional abaixo):** ao ler um recorte, o OCR devolve **todo** texto que encontra: nome do estado, slogan, cidade. Por isso um sistema real precisa de pós-processamento: ficar com o texto de caracteres mais altos (é o que a célula faz) e conferir se ele tem o formato de placa esperado. No Brasil, o padrão Mercosul é LLLNLNN (3 letras, 1 número, 1 letra, 2 números), o que dá para verificar com uma expressão regular.

**Pergunta para discussão:** em que condições você esperaria que até o YOLO ajustado errasse mais, e como isso deve aparecer no artigo?

**Resposta esperada:** placas pequenas (veículos distantes), noite, chuva, placas sujas ou muito inclinadas e motos. No artigo, o desempenho deve ser apresentado **por condição**, como fizemos em 7.3, e não só a média geral (cap. 15 e 24). E as imagens de teste devem vir de câmeras ou dias diferentes das de treino.

In [ ]:
# (Opcional, no Colab com internet) Ler o texto de uma placa com OCR, a partir do recorte da caixa anotada.
RODAR_OCR = False

if RODAR_OCR:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'easyocr'], check=True)
    import easyocr
    leitor = easyocr.Reader(['en'], gpu=False)
    grandes = res_v2.sort_values('tamanho da placa (% da foto)', ascending=False).head(6)   # placas grandes: OCR mais fácil
    fig, eixos = plt.subplots(2, 3, figsize=(13, 6))
    for ax, (_, linha) in zip(eixos.ravel(), grandes.iterrows()):
        im = next(i for i in imagens if i['file_name'] == linha.arquivo)
        x1, y1, x2, y2 = map(int, caixas_reais[im['id']][0])
        recorte = ler(im)[y1:y2, x1:x2]
        leituras = leitor.readtext(recorte, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-')
        # O OCR lê TUDO: nome do estado, slogan, cidade. Pós-processamento simples:
        # o texto principal da placa é o de caracteres mais altos (caixa de maior altura).
        altura = lambda r: max(p[1] for p in r[0]) - min(p[1] for p in r[0])
        principal = max(leituras, key=altura)[1] if leituras else '(nada)'
        tudo = ' | '.join(r[1] for r in leituras)
        ax.imshow(recorte); ax.axis('off')
        ax.set_title(f'principal: {principal}\ntudo: {tudo[:40]}{"…" if len(tudo) > 40 else ""}', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('OCR opcional desligado. No Colab, mude RODAR_OCR para True (precisa de internet).')

## Checklist
- [ ] Sei explicar por que imagem é matriz e o que um filtro de convolução faz.
- [ ] Vi o MLP falhar com deslocamento e a CNN resistir, e sei explicar por quê.
- [ ] Sei calcular IoU e interpretar precisão, recall e AP/mAP de um detector.
- [ ] Sei que, num projeto, o modelo pré-treinado é o baseline e o ajuste fino é o modelo a comparar.
- [ ] Sei que o split de vídeo é por câmera ou por dia, e que placas são dado pessoal (LGPD).
- [ ] Entendi, com números de fotos reais, por que regras escritas à mão falham e um detector treinado é necessário.